# Shabaka Pulse — NASA POWER Data Acquisition & Preparation

This notebook pulls hourly solar irradiance data for Benban Solar Park and wind data for the Gulf of Suez (Ras Ghareb region) from NASA's public POWER API, cleans it, and prepares a merged pandas DataFrame for model training.

Locations:
- Benban Solar Park (24.43°N, 32.74°E): Solar irradiance and temperature
- Ras Ghareb / Gulf of Suez (28.35°N, 33.08°E): Wind speed, direction, and temperature

Note: NASA POWER is a free public API and does not require an API key.

## 1. Setup

In [ ]:
from typing import Optional
import requests
import pandas as pd
import numpy as np
import time
import os
from pathlib import Path

# Create data directory for raw cached files and processed output
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. NASA POWER API client functions

The NASA POWER API provides hourly weather data at a single point coordinate.

Endpoint details:
- Base URL: https://power.larc.nasa.gov/api/temporal/hourly/point
- Authentication: None required
- Community: "RE" (Renewable Energy)

In [ ]:
NASA_POWER_ENDPOINT = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_power_data(
    latitude: float,
    longitude: float,
    parameters: list[str],
    start: str,
    end: str,
    community: str = "RE",
) -> dict:
    """Sends GET request to NASA POWER API and returns JSON response."""
    # Query parameters:
    # - parameters: comma-separated parameter names
    # - community: RE selects renewable energy variables
    # - format: json format
    params = {
        "parameters": ",".join(parameters),
        "community": community,
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "json",
    }

    response = requests.get(NASA_POWER_ENDPOINT, params=params, timeout=60)

    if response.status_code != 200:
        raise RuntimeError(
            f"API request failed with status code {response.status_code}.\n"
            f"URL: {response.url}\n"
            f"Details: {response.text[:500]}"
        )

    return response.json()


def parse_power_response(json_response: dict) -> pd.DataFrame:
    """Extracts parameters from JSON response and returns a DataFrame with DatetimeIndex."""
    param_data = json_response["properties"]["parameter"]
    series_dict: dict[str, pd.Series] = {}

    for param_name, time_map in param_data.items():
        idx = pd.to_datetime(list(time_map.keys()), format="%Y%m%d%H")
        vals = list(time_map.values())
        series_dict[param_name] = pd.Series(vals, index=idx, name=param_name)

    df = pd.DataFrame(series_dict)
    df.index.name = "timestamp_utc"

    # NASA POWER uses -999.0 as a sentinel value for missing data
    df.replace(-999.0, np.nan, inplace=True)

    return df


def fetch_and_parse(
    latitude: float,
    longitude: float,
    parameters: list[str],
    start: str,
    end: str,
    community: str = "RE",
    retries: int = 2,
    retry_wait: float = 5.0,
) -> pd.DataFrame:
    """Fetches and parses NASA POWER data with retry logic for network stability."""
    last_exception: Optional[Exception] = None

    for attempt in range(1, retries + 2):
        try:
            json_data = fetch_power_data(
                latitude, longitude, parameters, start, end, community
            )
            return parse_power_response(json_data)
        except (requests.RequestException, RuntimeError, KeyError) as e:
            last_exception = e
            if attempt <= retries:
                print(
                    f"Warning: Request attempt {attempt} failed ({e}). Retrying in {retry_wait}s..."
                )
                time.sleep(retry_wait)

    raise RuntimeError(
        f"Failed to fetch data after {retries + 1} attempts. Last error: {last_exception}"
    )

## 3. Smoke test — confirm the API is reachable

Fetching 1 week of Benban data to test connection and verify response format.

If this cell fails, check your network connection or proxy settings before proceeding.

In [ ]:
# Test request: 1 week of data for Benban Solar Park
BENBAN_LAT, BENBAN_LON = 24.43, 32.74

SOLAR_PARAMS = [
    "ALLSKY_SFC_SW_DWN",  # GHI: Global Horizontal Irradiance (W/m2)
    "ALLSKY_SFC_SW_DNI",  # DNI: Direct Normal Irradiance (W/m2)
    "T2M",                # Temperature at 2m (C)
    "CLRSKY_SFC_SW_DWN",  # Clear sky GHI (W/m2)
]

smoke_df = fetch_and_parse(
    latitude=BENBAN_LAT,
    longitude=BENBAN_LON,
    parameters=SOLAR_PARAMS,
    start="20250101",
    end="20250107",
)

smoke_df.info()
display(smoke_df.head())

## 4. Fetch full-year data for both locations

Pulling full-year hourly data for 2025 (2025-01-01 to 2025-12-31) to capture seasonal variation across both sites.

Locations:
- Solar: Benban (24.43°N, 32.74°E)
- Wind: Ras Ghareb / Gulf of Suez (28.35°N, 33.08°E)

In [ ]:
FORCE_REFETCH = False

START_DATE = "20250101"
END_DATE = "20251231"

RAW_BENBAN_PATH = DATA_DIR / "raw_benban.csv"
RAW_SUEZ_PATH = DATA_DIR / "raw_suez.csv"

# Wind parameters for Ras Ghareb
SUEZ_LAT, SUEZ_LON = 28.35, 33.08
WIND_PARAMS = [
    "WS10M",  # Wind speed at 10m (m/s)
    "WS50M",  # Wind speed at 50m (m/s)
    "WD50M",  # Wind direction at 50m (degrees)
    "T2M",    # Temperature at 2m (C)
]

# Benban solar data
if not FORCE_REFETCH and RAW_BENBAN_PATH.exists():
    df_benban = pd.read_csv(
        RAW_BENBAN_PATH, index_col="timestamp_utc", parse_dates=True
    )
else:
    df_benban = fetch_and_parse(
        latitude=BENBAN_LAT,
        longitude=BENBAN_LON,
        parameters=SOLAR_PARAMS,
        start=START_DATE,
        end=END_DATE,
    )
    df_benban.to_csv(RAW_BENBAN_PATH)

# Ras Ghareb wind data
if not FORCE_REFETCH and RAW_SUEZ_PATH.exists():
    df_suez = pd.read_csv(
        RAW_SUEZ_PATH, index_col="timestamp_utc", parse_dates=True
    )
else:
    df_suez = fetch_and_parse(
        latitude=SUEZ_LAT,
        longitude=SUEZ_LON,
        parameters=WIND_PARAMS,
        start=START_DATE,
        end=END_DATE,
    )
    df_suez.to_csv(RAW_SUEZ_PATH)

## 5. Merge and feature-engineer

Before merging:
- Rename `T2M` in each dataset to `t2m_benban` and `t2m_suez` to prevent column name collisions.
- Merge both datasets on the UTC timestamp index using an inner join.

Feature engineering steps:
- Extract `hour_of_day`, `month`, and `day_of_year` from the timestamp index.
- Calculate 100m wind speed (`ws100m`) using the power-law equation: `ws100m = WS50M * (100 / 50)^0.14` (approximating hub height wind speed since NASA POWER hourly provides 50m).

In [ ]:
# Rename temperature columns before merging
df_benban = df_benban.rename(columns={"T2M": "t2m_benban"})
df_suez = df_suez.rename(columns={"T2M": "t2m_suez"})

# Inner join on timestamp index
df = df_benban.join(df_suez, how="inner")

# Temporal features
df["hour_of_day"] = df.index.hour
df["month"] = df.index.month
df["day_of_year"] = df.index.dayofyear

# Wind speed extrapolation to 100m hub height (alpha = 0.14)
ALPHA = 0.14
df["ws100m"] = df["WS50M"] * (100 / 50) ** ALPHA

# Standardize column names to lowercase
df.columns = [c.lower() for c in df.columns]

## 6. Data quality check

Checking summary statistics, index range, and missing value percentages across columns.

In [ ]:
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Total rows: {len(df):,}")

missing_pct = (df.isna().sum() / len(df)) * 100
missing_df = pd.DataFrame(
    {"missing_count": df.isna().sum(), "missing_pct": missing_pct.round(2)}
)
display(missing_df)

high_missing = missing_pct[missing_pct > 5.0]
if not high_missing.empty:
    print("Warning: Columns with >5% missing data:")
    for col, pct in high_missing.items():
        print(f"  - {col}: {pct:.2f}%")

key_columns = [
    c
    for c in df.columns
    if any(k in c for k in ["allsky", "clrsky", "ws", "wd"])
]
display(df[key_columns].describe().round(2))

## 7. Save final dataset

Saving the processed DataFrame to `data/nasa_power_training_data.csv`.

In [ ]:
OUTPUT_PATH = DATA_DIR / "nasa_power_training_data.csv"
df.to_csv(OUTPUT_PATH)

print(f"Saved dataset to {OUTPUT_PATH} ({df.shape[0]:,} rows, {df.shape[1]} columns)")
display(df.head())

## 8. CSV File Parameters & Row Count Summary

Overview of each CSV file generated by this notebook, including location, parameter definitions, and row counts:

### 1. `data/raw_benban.csv`
- **Location**: Benban Solar Park (24.43°N, 32.74°E)
- **Rows**: 8,760 hourly records (full year 2025: 365 days × 24 hours)
- **Parameters**:
  - `timestamp_utc`: Hourly UTC datetime index (`YYYY-MM-DD HH:00:00`)
  - `ALLSKY_SFC_SW_DWN`: All-sky Global Horizontal Irradiance (GHI) [W/m²]
  - `ALLSKY_SFC_SW_DNI`: Direct Normal Irradiance (DNI) [W/m²]
  - `CLRSKY_SFC_SW_DWN`: Clear-sky GHI (theoretical maximum irradiance) [W/m²]
  - `T2M`: Air temperature at 2 meters above ground [°C]

### 2. `data/raw_suez.csv`
- **Location**: Ras Ghareb / Gulf of Suez (28.35°N, 33.08°E)
- **Rows**: 8,760 hourly records (full year 2025: 365 days × 24 hours)
- **Parameters**:
  - `timestamp_utc`: Hourly UTC datetime index (`YYYY-MM-DD HH:00:00`)
  - `WS10M`: Wind speed at 10 meters height [m/s]
  - `WS50M`: Wind speed at 50 meters height [m/s]
  - `WD50M`: Wind direction at 50 meters height [degrees] (0° = North, 90° = East)
  - `T2M`: Air temperature at 2 meters above ground [°C]

### 3. `data/nasa_power_training_data.csv`
- **Purpose**: Cleaned, merged training dataset for solar/wind forecasting models
- **Rows**: 8,760 hourly records (inner join matching UTC timestamps across both locations)
- **Parameters** (13 total columns including index):
  - `timestamp_utc`: Hourly UTC datetime index
  - `allsky_sfc_sw_dwn`: Benban GHI irradiance [W/m²]
  - `allsky_sfc_sw_dni`: Benban DNI irradiance [W/m²]
  - `clrsky_sfc_sw_dwn`: Benban clear-sky GHI [W/m²]
  - `t2m_benban`: Benban 2m temperature [°C]
  - `ws10m`: Gulf of Suez 10m wind speed [m/s]
  - `ws50m`: Gulf of Suez 50m wind speed [m/s]
  - `wd50m`: Gulf of Suez 50m wind direction [degrees]
  - `t2m_suez`: Gulf of Suez 2m temperature [°C]
  - `hour_of_day`: Hour feature (0–23)
  - `month`: Month feature (1–12)
  - `day_of_year`: Day of year feature (1–365)
  - `ws100m`: Extrapolated 100m hub-height wind speed [m/s] (wind power law, $\alpha=0.14$)

In [ ]:
# Code summary verifying row counts and column parameters for all CSV files
csv_files = [
    ("Benban Solar Raw", DATA_DIR / "raw_benban.csv"),
    ("Suez Wind Raw", DATA_DIR / "raw_suez.csv"),
    ("Merged Training Dataset", DATA_DIR / "nasa_power_training_data.csv"),
]

summary_rows = []
for label, path in csv_files:
    if path.exists():
        temp_df = pd.read_csv(path)
        summary_rows.append(
            {
                "dataset": label,
                "file_name": path.name,
                "row_count": len(temp_df),
                "num_columns": len(temp_df.columns),
                "columns": ", ".join(temp_df.columns),
            }
        )

display(pd.DataFrame(summary_rows))

## Next step

This DataFrame is now ready to feed into the XGBoost solar/wind forecasting models (see `train_forecast_model` notebook).